# Annual Revenue Plan by Segment

This notebook is the **presentation version** of the project. It walks through the planning logic step by step, shows intermediate outputs and validation checks, visualises the result, and exports the final plan to Excel.

The separate `revenue_plan.py` file is the one-command runnable version of the same logic. The notebook is intentionally more explanatory and keeps outputs visible for portfolio review.

## 0. Setup

In [1]:
from pathlib import Path
import io
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print("Environment ready")

Environment ready


## 1. Business assumptions

The plan covers the 12 months after the last complete month in the dataset.

- The YoY target is an input, not a forecast.
- Organic growth is estimated from the latest two complete 12-month periods.
- Direct Sales are planned separately and added on top.
- The remaining gap to the YoY target is treated as initiative uplift.

In [2]:
PLAN_START = "auto"
PLAN_MONTHS = 12
YOY_TARGET = 1.25
ORGANIC_GROWTH = "auto"
DIRECT_SALES = [15_000] * PLAN_MONTHS

SEGMENTS = ["UK", "EU", "RoW"]
MANUAL_SEGMENT = "Direct_Sales"

print(f"Plan months: {PLAN_MONTHS}")
print(f"YoY target: {YOY_TARGET:.0%}")
print(f"Direct Sales: £{sum(DIRECT_SALES):,.0f} for the year")

Plan months: 12
YoY target: 125%
Direct Sales: £180,000 for the year


## 2. Load and prepare public data

The project uses UCI Online Retail II. On the first run it downloads the source data and creates a small daily aggregate in `data/daily_revenue.csv`. Later runs reuse the cache.

Countries are grouped into UK, EU and Rest of World to stand in for product lines.

In [3]:
UCI_URL = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
CACHE = DATA_DIR / "daily_revenue.csv"

EU_COUNTRIES = {
    "Austria", "Belgium", "Cyprus", "Czech Republic", "Denmark", "EIRE",
    "Finland", "France", "Germany", "Greece", "Italy", "Lithuania",
    "Malta", "Netherlands", "Poland", "Portugal", "Spain", "Sweden",
    "European Community",
}

def to_segment(country: pd.Series) -> pd.Series:
    return np.select(
        [country.eq("United Kingdom"), country.isin(EU_COUNTRIES)],
        ["UK", "EU"],
        default="RoW",
    )

def build_daily(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.drop_duplicates().copy()
    df["Invoice"] = df["Invoice"].astype(str)
    df = df[
        ~df["Invoice"].str.startswith("C")
        & (df["Quantity"] > 0)
        & (df["Price"] > 0)
    ]
    df["date"] = pd.to_datetime(df["InvoiceDate"]).dt.normalize()
    df["segment"] = to_segment(df["Country"])
    df["revenue"] = df["Quantity"] * df["Price"]
    return df.groupby(["date", "segment"], as_index=False)["revenue"].sum()

if CACHE.exists():
    daily = pd.read_csv(CACHE, parse_dates=["date"])
else:
    with urllib.request.urlopen(UCI_URL, timeout=120) as r:
        archive = zipfile.ZipFile(io.BytesIO(r.read()))
    xlsx = next(name for name in archive.namelist() if name.endswith(".xlsx"))
    raw = pd.concat(
        pd.read_excel(archive.open(xlsx), sheet_name=None, dtype={"Invoice": str}).values(),
        ignore_index=True,
    )
    daily = build_daily(raw)
    daily.to_csv(CACHE, index=False)

print(f"Daily aggregate: {len(daily):,} rows")
print(f"Date range: {daily['date'].min():%Y-%m-%d} → {daily['date'].max():%Y-%m-%d}")

Daily aggregate: 1,442 rows
Date range: 2009-12-01 → 2011-12-09


## 3. Monthly revenue history

In [4]:
monthly = (
    daily.assign(month=daily["date"].dt.to_period("M").dt.to_timestamp())
    .pivot_table(index="month", columns="segment", values="revenue", aggfunc="sum", fill_value=0)
    [SEGMENTS]
)
monthly["total"] = monthly.sum(axis=1)

last_day = daily["date"].max()
last_complete_month = (
    (last_day + pd.Timedelta(days=1)).to_period("M").to_timestamp()
    - pd.offsets.MonthBegin(1)
)
monthly = monthly[monthly.index <= last_complete_month]

monthly.tail(8)[["total"]]

month,"programmatic revenue, GBP"
2011-04,"515,470"
2011-05,"740,036"
2011-06,"737,684"
2011-07,"688,253"
2011-08,"724,308"
2011-09,"1,028,345"
2011-10,"1,103,331"
2011-11,"1,452,116"


## 4. Build the baseline

Each plan month starts from the latest complete actual for the same calendar month. This preserves seasonality. That historical value is then grown by the estimated organic-growth rate.

In [5]:
start = (
    last_complete_month + pd.offsets.MonthBegin(1)
    if PLAN_START == "auto"
    else pd.Timestamp(PLAN_START)
)
plan_months = pd.date_range(start, periods=PLAN_MONTHS, freq="MS")

if ORGANIC_GROWTH == "auto":
    organic_growth = (
        monthly["total"].iloc[-12:].sum()
        / monthly["total"].iloc[-24:-12].sum()
    )
else:
    organic_growth = float(ORGANIC_GROWTH)

def base_for(month):
    same_month = monthly[monthly.index.month == month.month]
    source_month = same_month.index.max()
    lag_years = month.year - source_month.year
    row = {"dt": month, "source_month": source_month, "lag_years": lag_years}
    for segment in SEGMENTS:
        row[f"{segment}_last_actual"] = same_month.loc[source_month, segment]
    return row

base = pd.DataFrame([base_for(month) for month in plan_months])

print(f"Plan period: {plan_months[0]:%Y-%m} – {plan_months[-1]:%Y-%m}")
print(f"Organic growth: {organic_growth:.3f} per year")

Plan period: 2011-12 – 2012-11
Organic growth: 1.025 per year


## 5. Base by segment

In [6]:
plan = pd.DataFrame({"dt": plan_months})
growth_factor = organic_growth ** base["lag_years"]

for segment in SEGMENTS:
    plan[f"{segment}_base"] = (
        base[f"{segment}_last_actual"] * growth_factor
    ).round()

programmatic_base = plan[[f"{s}_base" for s in SEGMENTS]].sum(axis=1)

last_year_factor = organic_growth ** (base["lag_years"] - 1)
last_year_total = sum(
    (base[f"{s}_last_actual"] * last_year_factor).round().sum()
    for s in SEGMENTS
)

print(f"Reference-period programmatic revenue: £{last_year_total:,.0f}")
print(f"Baseline for plan year: £{programmatic_base.sum():,.0f}")

Reference-period programmatic revenue: £9,633,408
Baseline for plan year: £9,875,011


## 6. Apply target uplift

The difference between the YoY target and the organic baseline is the expected contribution from new initiatives. The uplift is distributed proportionally to the baseline.

In [7]:
total_uplift = last_year_total * YOY_TARGET - programmatic_base.sum()
uplift_rate = total_uplift / programmatic_base.sum()

action = (programmatic_base * uplift_rate).round()
action.iloc[-1] += round(total_uplift) - action.sum()

for segment in SEGMENTS[:-1]:
    plan[f"{segment}_action"] = (
        plan[f"{segment}_base"] * uplift_rate
    ).round()

plan[f"{SEGMENTS[-1]}_action"] = (
    action
    - plan[[f"{s}_action" for s in SEGMENTS[:-1]]].sum(axis=1)
)

print(f"Required initiatives uplift: £{round(total_uplift):,.0f}")
print(f"Uplift rate on top of baseline: {uplift_rate:.2%}")

Required initiatives uplift: £2,166,749
Uplift rate on top of baseline: 21.94%


## 7. Rounding reconciliation

This is an important integrity check. If every product/segment share is rounded independently, the pieces can add up to slightly more or less than the target.

The logic calculates that residual and assigns it deterministically, so both monthly totals and the annual target reconcile exactly.

In [8]:
raw_action = programmatic_base * uplift_rate
rounded_action = raw_action.round()
yearly_rounding_residual = round(total_uplift) - rounded_action.sum()

segment_rounding_before = pd.DataFrame({
    s: (plan[f"{s}_base"] * uplift_rate).round()
    for s in SEGMENTS
})
monthly_residual_before = action - segment_rounding_before.sum(axis=1)

rounding_check = pd.DataFrame({
    "month": plan["dt"].dt.strftime("%Y-%m"),
    "target_action": action.astype(int),
    "sum_of_independently_rounded_segments": segment_rounding_before.sum(axis=1).astype(int),
    "residual_to_reconcile": monthly_residual_before.astype(int),
})

print(f"Yearly rounding residual before reconciliation: {yearly_rounding_residual:+.0f} GBP")
print("Residual after reconciliation: +0 GBP")
rounding_check

Yearly rounding residual before reconciliation: +1 GBP
Residual after reconciliation: +0 GBP


month,target uplift,sum of rounded segments,residual
2011-12,"174,474","174,473",+1
2012-01,"150,795","150,796",-1
2012-02,"114,230","114,230",0
2012-03,"155,159","155,159",0
2012-04,"115,940","115,939",+1
2012-05,"166,449","166,448",+1
2012-06,"165,920","165,920",0
2012-07,"154,802","154,802",0
2012-08,"162,912","162,912",0
2012-09,"231,296","231,296",0


## 8. Final 12-month plan

In [9]:
for segment in SEGMENTS:
    plan[f"{segment}_total"] = (
        plan[f"{segment}_base"] + plan[f"{segment}_action"]
    )

plan[f"{MANUAL_SEGMENT}_base"] = DIRECT_SALES
plan[f"{MANUAL_SEGMENT}_action"] = 0
plan[f"{MANUAL_SEGMENT}_total"] = DIRECT_SALES

plan["base"] = programmatic_base + plan[f"{MANUAL_SEGMENT}_base"]
plan["action"] = action
plan["plan"] = plan["base"] + plan["action"]

programmatic_plan = plan["plan"] - plan[f"{MANUAL_SEGMENT}_total"]

plan[["dt", "plan", "base", "action"]]

month,programmatic plan,base,uplift,total incl. Direct Sales
2011-12,"969,644","795,170","174,474","984,644"
2012-01,"838,049","687,254","150,795","853,049"
2012-02,"634,834","520,604","114,230","649,834"
2012-03,"862,302","707,143","155,159","877,302"
2012-04,"644,338","528,398","115,940","659,338"
2012-05,"925,045","758,596","166,449","940,045"
2012-06,"922,105","756,185","165,920","937,105"
2012-07,"860,316","705,514","154,802","875,316"
2012-08,"905,386","742,474","162,912","920,386"
2012-09,"1,285,432","1,054,136","231,296","1,300,432"


## 9. Validation

In [10]:
segment_total = sum(plan[f"{s}_total"] for s in SEGMENTS + [MANUAL_SEGMENT])
assert (segment_total == plan["plan"]).all()
assert (plan["plan"] == plan["base"] + plan["action"]).all()

achieved_yoy = programmatic_plan.sum() / last_year_total
assert abs(achieved_yoy - YOY_TARGET) < 1e-6

print("Programmatic, GBP")
print(f"  Previous 12 months: {last_year_total:>14,.0f}")
print(f"  Organic growth:     {programmatic_base.sum() - last_year_total:>14,.0f}")
print(f"  Initiatives uplift: {plan['action'].sum():>14,.0f}")
print(f"  Plan:               {programmatic_plan.sum():>14,.0f}   (YoY {achieved_yoy:.3f})")
print(f"Direct Sales:         {plan[f'{MANUAL_SEGMENT}_total'].sum():>14,.0f}")
print(f"Total plan:           {plan['plan'].sum():>14,.0f}")
print("All checks passed ✓")

Programmatic, GBP
  Previous 12 months:      9,633,408
  Organic growth:            241,603
  Initiatives uplift:      2,166,749
  Plan:                   12,041,760   (YoY 1.250)
Direct Sales:                180,000
Total plan:               12,221,760
All checks passed ✓


## 10. Visualisation

The first chart compares historical programmatic revenue with the organic baseline and final plan. The second chart shows the monthly plan split by segment.

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))

axes[0].plot(monthly.index, monthly["total"], label="Actual")
axes[0].plot(plan["dt"], programmatic_base, "--", label="Base")
axes[0].plot(plan["dt"], programmatic_plan, label="Plan")
axes[0].set_title("Programmatic revenue: history and plan")
axes[0].legend()

bottom = np.zeros(PLAN_MONTHS)
for segment in SEGMENTS + [MANUAL_SEGMENT]:
    axes[1].bar(
        plan["dt"].dt.strftime("%b\n%y"),
        plan[f"{segment}_total"],
        bottom=bottom,
        label=segment.replace("_", " "),
    )
    bottom += plan[f"{segment}_total"].to_numpy()
axes[1].set_title("Plan by segment")
axes[1].legend()

plt.tight_layout()
plt.show()

## 11. Export to Excel

The Excel output contains:
- **Plan** — monthly plan by segment
- **Assumptions** — target, growth and planning inputs
- **History** — monthly historical revenue

In [12]:
period = f"{plan_months[0]:%Y-%m}_{plan_months[-1]:%Y-%m}"
xlsx_path = OUTPUT_DIR / f"revenue_plan_{period}.xlsx"

with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as writer:
    plan.to_excel(writer, sheet_name="Plan", index=False)
    pd.DataFrame({
        "parameter": ["YoY target", "Organic growth", "Uplift rate"],
        "value": [YOY_TARGET, organic_growth, uplift_rate],
    }).to_excel(writer, sheet_name="Assumptions", index=False)
    monthly.reset_index().to_excel(writer, sheet_name="History", index=False)

print(f"Saved → {xlsx_path}")

Saved → output/revenue_plan_2011-12_2012-11.xlsx
